<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/analise_final/CONSTRU%C3%87%C3%83O_DO_NOVO_R%C3%93TULO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construção do Novo Rótulo de Classificação

Após a realização dos experimentos iniciais com quatro classes, observou-se que os modelos apresentavam dificuldade recorrente em diferenciar as categorias **Atenção** e **Crítica**. Essa dificuldade apareceu de forma consistente em diferentes cenários, incluindo testes com balanceamento automático, pesos manuais, recortes temporais, redução e ajuste de hiperparâmetros.

Mesmo quando houve melhora no desempenho geral dos modelos, as matrizes de confusão continuaram indicando que grande parte dos erros estava concentrada justamente entre essas duas classes. Ou seja, muitas amostras originalmente classificadas como **Crítica** eram previstas como **Atenção**, e vice-versa.

A análise da construção do rótulo mostrou que essa dificuldade não estava relacionada apenas ao algoritmo utilizado, mas também à própria estrutura das classes. O rótulo original era construído a partir de um score ambiental baseado em cinco parâmetros:

* pH;
* Oxigênio dissolvido;
* DBO;
* Nitrato;
* Amônia.

Cada parâmetro recebia valor 1 quando estava dentro do limite definido e 0 quando estava fora do limite. A soma desses valores gerava um score de 0 a 5, posteriormente convertido nas classes:

```text
Score 5 → Excelente
Score 4 → Boa
Score 3 → Atenção
Score 0, 1 ou 2 → Crítica
```

No entanto, ao analisar os resultados, percebeu-se que as classes **Atenção** e **Crítica** estavam muito próximas entre si. Na prática, a diferença entre essas categorias podia depender da alteração de apenas um parâmetro no score. Isso tornava a fronteira entre as duas classes muito sensível e difícil de ser aprendida pelos modelos, principalmente porque as variáveis usadas para construir o rótulo não foram incluídas diretamente no treinamento, a fim de evitar *data leakage*.

Foram testadas diferentes alternativas para tentar ajustar essa separação, como alterações nos critérios da classe Crítica e diferentes formas de balanceamento. Porém, nenhuma dessas soluções apresentou uma melhora consistente sem gerar novos problemas, como redução excessiva da quantidade de amostras críticas ou aumento artificial dos falsos positivos.

Diante disso, optou-se por reformular o rótulo em três classes, agrupando as categorias **Atenção** e **Crítica** em uma nova classe chamada **Não adequada**. A nova estrutura passa a ser:

```text
Score 5 → Adequada
Score 4 → Boa
Score 0, 1, 2 ou 3 → Não adequada
```

Essa decisão foi tomada porque, para o objetivo do sistema AquaSense, o modelo não precisa atuar como diagnóstico final da condição ambiental da água. O papel do modelo é funcionar como uma ferramenta de triagem inicial, auxiliando o gestor a identificar registros que indicam possível inadequação e que precisam ser avaliados com maior atenção por uma equipe técnica.

Assim, a classe **Não adequada** representa situações em que os parâmetros analisados indicam algum nível de comprometimento da qualidade da água, mas sem afirmar de forma definitiva que o corpo hídrico está em condição crítica. Essa abordagem é mais compatível com a proposta do sistema, pois permite automatizar uma primeira análise dos registros e apoiar a tomada de decisão, sem substituir a validação técnica especializada.

Além disso, a nova classificação reduz a sobreposição entre classes, aumenta a representatividade da menor categoria e torna o problema de classificação mais coerente com as informações disponíveis no conjunto de dados.

Portanto, a construção do novo rótulo em três classes foi adotada como uma solução metodológica mais consistente, considerando tanto os resultados dos experimentos quanto o contexto prático de aplicação do AquaSense.


In [1]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/water_quality_2000_2008.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 23)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Excelente,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Excelente,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Boa,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Boa,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Boa,2003


In [4]:
df.columns.tolist()

['Country',
 'Area',
 'Waterbody Type',
 'Date',
 'Ammonia (mg/l)',
 'Biochemical Oxygen Demand (mg/l)',
 'Dissolved Oxygen (mg/l)',
 'Orthophosphate (mg/l)',
 'pH (ph units)',
 'Temperature (cel)',
 'Nitrogen (mg/l)',
 'Nitrate (mg/l)',
 'CCME_Values',
 'CCME_WQI',
 'ph_ok',
 'od_ok',
 'dbo_ok',
 'nitrate_ok',
 'ammonia_limit',
 'ammonia_ok',
 'environmental_score',
 'conama_status',
 'Year']

In [5]:
df["conama_status"].value_counts()

,count
conama_status,
Excelente,41169
Boa,11796
Atenção,5842
Crítica,1089


In [7]:
df["conama_status"].value_counts(normalize=True).round(4)

,proportion
conama_status,
Excelente,0.6873
Boa,0.1969
Atenção,0.0975
Crítica,0.0182


In [8]:
df_ac = df[df["conama_status"].isin(["Atenção", "Crítica"])]

df_ac["conama_status"].value_counts()

,count
conama_status,
Atenção,5842
Crítica,1089


In [9]:
df_ac.groupby("conama_status").describe()

Date                                                      \
              count                           mean                  min   
conama_status                                                             
Atenção        5842  2004-04-29 13:31:12.098596352  2000-01-05 00:00:00   
Crítica        1089  2004-06-09 19:13:03.471074560  2000-01-06 00:00:00   

                                                                              \
                               25%                  50%                  75%   
conama_status                                                                  
Atenção        2002-04-19 18:00:00  2004-04-16 00:00:00  2006-03-28 00:00:00   
Crítica        2002-07-31 00:00:00  2004-06-03 00:00:00  2006-05-23 00:00:00   

                                        Ammonia (mg/l)             ...  \
                               max  std          count       mean  ...   
conama_status                                                      ...   
Atenção        2008-12-29 00:00:00  NaN         5842.0  10.883500  ...   
Crítica        2008-12-15 00:00:00  NaN         1089.0   8.115326  ...   

              environmental_score              Year                       \
                              max       std   count         mean     min   
conama_status                                                              
Atenção                       3.0  0.000000  5842.0  2003.850565  2000.0   
Crítica                       2.0  0.100041  1089.0  2003.965106  2000.0   

                                                         
                  25%     50%     75%     max       std  
conama_status                                            
Atenção        2002.0  2004.0  2006.0  2008.0  2.436685  
Crítica        2002.0  2004.0  2006.0  2008.0  2.397278  

[2 rows x 144 columns]

In [11]:
numeric_cols = df_ac.select_dtypes(include="number").columns

df_ac.groupby("conama_status")[numeric_cols].mean()

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,Year
conama_status,,,,,,,,,,,,,,,,,
Atenção,10.883500,26.192542,10.063253,3.830632,7.595184,12.506699,11.136436,8.036063,50.253178,0.979973,0.978261,0.056145,0.724409,2.522133,0.261212,3.000000,2003.850565
Crítica,8.115326,17.578127,9.771335,4.801740,7.601419,12.686263,17.080223,16.737704,46.848063,0.928375,0.930211,0.008264,0.093664,2.417906,0.029385,1.989899,2003.965106


In [12]:
df_ac.groupby("conama_status")[numeric_cols].median()

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,Year
conama_status,,,,,,,,,,,,,,,,,
Atenção,4.96,10.8,10.2,3.05,7.67,11.46,7.7,4.5,45.735118,1.0,1.0,0.0,1.0,2.0,0.0,3.0,2004.0
Crítica,4.95,11.8,10.2,4.57,7.75,11.46,16.0,15.5,44.048838,1.0,1.0,0.0,0.0,2.0,0.0,2.0,2004.0


In [13]:
df_ac.groupby("conama_status")[numeric_cols].std()

,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,Year
conama_status,,,,,,,,,,,,,,,,,
Atenção,15.187942,43.283874,1.373848,4.075223,0.556563,3.809355,9.670032,7.874185,11.530979,0.140106,0.145843,0.230222,0.446850,0.943122,0.439333,0.000000,2.436685
Crítica,12.360228,25.446339,2.008815,4.575986,0.897748,4.145933,8.420101,7.825889,8.406049,0.257985,0.254908,0.090574,0.291494,0.941819,0.168960,0.100041,2.397278


In [14]:
comparacao = pd.DataFrame({
    "media_atencao": df_ac[df_ac["conama_status"] == "Atenção"][numeric_cols].mean(),
    "media_critica": df_ac[df_ac["conama_status"] == "Crítica"][numeric_cols].mean(),
    "mediana_atencao": df_ac[df_ac["conama_status"] == "Atenção"][numeric_cols].median(),
    "mediana_critica": df_ac[df_ac["conama_status"] == "Crítica"][numeric_cols].median()
})

comparacao["dif_media"] = abs(comparacao["media_critica"] - comparacao["media_atencao"])

comparacao.sort_values("dif_media", ascending=False)

,media_atencao,media_critica,mediana_atencao,mediana_critica,dif_media
Nitrate (mg/l),8.036063,16.737704,4.500000,15.500000,8.701641
Biochemical Oxygen Demand (mg/l),26.192542,17.578127,10.800000,11.800000,8.614415
Nitrogen (mg/l),11.136436,17.080223,7.700000,16.000000,5.943787
CCME_Values,50.253178,46.848063,45.735118,44.048838,3.405115
Ammonia (mg/l),10.883500,8.115326,4.960000,4.950000,2.768175
environmental_score,3.000000,1.989899,3.000000,2.000000,1.010101
Orthophosphate (mg/l),3.830632,4.801740,3.050000,4.570000,0.971108
nitrate_ok,0.724409,0.093664,1.000000,0.000000,0.630746
Dissolved Oxygen (mg/l),10.063253,9.771335,10.200000,10.200000,0.291918
ammonia_ok,0.261212,0.029385,0.000000,0.000000,0.231827


In [15]:
df["environmental_score_original"] = df["environmental_score"]
df["conama_status_original"] = df["conama_status"]

In [16]:
aux_cols = ["ph_ok", "od_ok", "dbo_ok", "nitrate_ok", "ammonia_ok"]

df[aux_cols].head()

,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_ok
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,0,1,1
3,1,1,1,0,1
4,1,1,1,0,1


In [17]:
df["environmental_score_adjusted"] = (
    df["ph_ok"] +
    df["od_ok"] +
    df["dbo_ok"] +
    df["nitrate_ok"] +
    df["ammonia_ok"]
)

In [18]:
def classify_conama_adjusted(row):
    score = row["environmental_score_adjusted"]

    # Crítica apenas se tiver score <= 2
    # e falhar em pelo menos um parâmetro sensível
    if score <= 2 and (
        row["nitrate_ok"] == 0 or
        row["od_ok"] == 0 or
        row["dbo_ok"] == 0
    ):
        return "Crítica"

    elif score <= 3:
        return "Atenção"

    elif score == 4:
        return "Boa"

    else:
        return "Excelente"

In [19]:
df["conama_status_adjusted"] = df.apply(
    classify_conama_adjusted,
    axis=1
)

In [20]:
print("Rótulo original:")
print(df["conama_status_original"].value_counts())

print("\nRótulo ajustado:")
print(df["conama_status_adjusted"].value_counts())

Rótulo original:
conama_status_original
Excelente    41169
Boa          11796
Atenção       5842
Crítica       1089
Name: count, dtype: int64

Rótulo ajustado:
conama_status_adjusted
Excelente    41169
Boa          11796
Atenção       5842
Crítica       1089
Name: count, dtype: int64


In [21]:
print("Rótulo original (%):")
print(df["conama_status_original"].value_counts(normalize=True).round(4))

print("\nRótulo ajustado (%):")
print(df["conama_status_adjusted"].value_counts(normalize=True).round(4))

Rótulo original (%):
conama_status_original
Excelente    0.6873
Boa          0.1969
Atenção      0.0975
Crítica      0.0182
Name: proportion, dtype: float64

Rótulo ajustado (%):
conama_status_adjusted
Excelente    0.6873
Boa          0.1969
Atenção      0.0975
Crítica      0.0182
Name: proportion, dtype: float64


In [22]:
df["critical_failures"] = (
    (df["nitrate_ok"] == 0).astype(int) +
    (df["od_ok"] == 0).astype(int) +
    (df["dbo_ok"] == 0).astype(int)
)

In [23]:
def classify_conama_adjusted(row):
    score = row["environmental_score"]

    if score <= 2 and row["critical_failures"] >= 2:
        return "Crítica"

    elif score <= 3:
        return "Atenção"

    elif score == 4:
        return "Boa"

    else:
        return "Excelente"

df["conama_status_adjusted"] = df.apply(classify_conama_adjusted, axis=1)

In [24]:
pd.crosstab(
    df["conama_status"],
    df["conama_status_adjusted"]
)

conama_status_adjusted,Atenção,Boa,Crítica,Excelente
conama_status,,,,
Atenção,5842,0,0,0
Boa,0,11796,0,0
Crítica,56,0,1033,0
Excelente,0,0,0,41169


In [25]:
df["conama_status_adjusted"].value_counts()
df["conama_status_adjusted"].value_counts(normalize=True).round(4)

,proportion
conama_status_adjusted,
Excelente,0.6873
Boa,0.1969
Atenção,0.0985
Crítica,0.0172


#teste 2

In [26]:
df["conama_status_original"] = df["conama_status"]

In [27]:
def classify_v1(score):

    if score == 5:
        return "Excelente"

    elif score == 4:
        return "Boa"

    elif score in [2, 3]:
        return "Atenção"

    else:
        return "Crítica"

In [28]:
df["conama_status_v1"] = df["environmental_score"].apply(classify_v1)

In [29]:
df["conama_status_v1"].value_counts()

,count
conama_status_v1,
Excelente,41169
Boa,11796
Atenção,6920
Crítica,11


In [30]:
df["conama_status_v1"].value_counts(
    normalize=True
).round(4)

,proportion
conama_status_v1,
Excelente,0.6873
Boa,0.1969
Atenção,0.1155
Crítica,0.0002


In [31]:
pd.crosstab(
    df["conama_status_original"],
    df["conama_status_v1"]
)

conama_status_v1,Atenção,Boa,Crítica,Excelente
conama_status_original,,,,
Atenção,5842,0,0,0
Boa,0,11796,0,0
Crítica,1078,0,11,0
Excelente,0,0,0,41169


In [32]:
df["environmental_score"].value_counts().sort_index()

,count
environmental_score,
1,11
2,1078
3,5842
4,11796
5,41169


In [33]:
df[df["conama_status"] == "Crítica"][
    ["ph_ok","od_ok","dbo_ok","nitrate_ok","ammonia_ok"]
]

,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_ok
1126,1,1,0,0,0
1192,1,1,0,0,0
1213,1,1,0,0,0
1246,1,1,0,0,0
1260,1,1,0,0,0
...,...,...,...,...,...
55198,1,1,0,0,0
55326,1,1,0,0,0
55336,1,1,0,0,0
55534,1,1,0,0,0


In [34]:
criticas = df[df["conama_status"] == "Crítica"]

criticas.groupby(
    ["ph_ok","od_ok","dbo_ok","nitrate_ok","ammonia_ok"]
).size().sort_values(ascending=False)

ph_ok  od_ok  dbo_ok  nitrate_ok  ammonia_ok
1      1      0       0           0             942
0      1      0       1           0              51
1      0      0       1           0              45
                      0           1              15
0      1      0       0           1              11
1      0      0       0           0               6
0      1      0       0           0               5
       0      0       1           1               5
       1      1       0           0               4
1      0      1       0           0               3
0      0      1       1           0               1
                      0           1               1
dtype: int64

In [35]:
cols = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)"
]

df[df["conama_status"].isin(["Atenção","Crítica"])] \
.groupby("conama_status")[cols] \
.mean()

,Temperature (cel),Orthophosphate (mg/l),Nitrogen (mg/l)
conama_status,,,
Atenção,12.506699,3.830632,11.136436
Crítica,12.686263,4.801740,17.080223


In [51]:
def classify_v2(row):
    score = row["environmental_score"]

    if (
        row["ph_ok"] == 1 and
        row["od_ok"] == 1 and
        row["dbo_ok"] == 0 and
        row["nitrate_ok"] == 0 and
        row["ammonia_ok"] == 0
    ):
        return "Crítica"

    elif score <= 3:
        return "Atenção"

    elif score == 4:
        return "Boa"

    else:
        return "Excelente"

df["conama_status_v2"] = df.apply(classify_v2, axis=1)

In [52]:
df["conama_status_v2"].value_counts()
df["conama_status_v2"].value_counts(normalize=True).round(4)

,proportion
conama_status_v2,
Excelente,0.6873
Boa,0.1969
Atenção,0.1000
Crítica,0.0157


In [53]:
pd.crosstab(df["conama_status"], df["conama_status_v2"])

conama_status_v2,Atenção,Boa,Crítica,Excelente
conama_status,,,,
Atenção,5842,0,0,0
Boa,0,11796,0,0
Crítica,147,0,942,0
Excelente,0,0,0,41169


In [69]:
def classify_v3(row):
    score = row["environmental_score"]

    if score <= 2 and row["dbo_ok"] == 0 and row["nitrate_ok"] == 0:
        return "Crítica"

    elif score <= 3:
        return "Atenção"

    elif score == 4:
        return "Boa"

    else:
        return "Excelente"

df["conama_status_v3"] = df.apply(classify_v3, axis=1)

# teste 4

In [89]:
df["environmental_score"] = (
    df["ph_ok"] +
    df["od_ok"] +
    df["dbo_ok"] +
    df["nitrate_ok"] +
    df["ammonia_ok"]
)

In [90]:
df["conama_status_original"] = df["conama_status"]

In [91]:
def classify_conama_v4(score):

    if score == 5:
        return "Adequada"

    elif score == 4:
        return "Boa"

    else:
        return "Não adequada"

In [92]:
df["conama_status_v4"] = df["environmental_score"].apply(
    classify_conama_v4
)

In [93]:
df["conama_status_v4"].value_counts()

,count
conama_status_v4,
Adequada,41169
Boa,11796
Não adequada,6931


In [94]:
df["conama_status_v4"].value_counts(
    normalize=True
).round(4)

,proportion
conama_status_v4,
Adequada,0.6873
Boa,0.1969
Não adequada,0.1157


In [96]:
pd.crosstab(
    df["conama_status_original"],
    df["conama_status_v4"]
)

conama_status_v4,Adequada,Boa,Não adequada
conama_status_original,,,
Atenção,0,0,5842
Boa,0,11796,0
Crítica,0,0,1089
Excelente,41169,0,0


In [97]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status_v4"]

In [98]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [99]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [84]:
# teste 1
class_weights = {
    "Excelente": 1,
    "Boa": 2,
    "Atenção": 4,
    "Crítica": 10
}


In [126]:
class_weights = {
    "Adequada": 1,
    "Boa": 2,
    "Não adequada": 2
}

In [127]:
# com balanceamento manual
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                class_weight=class_weights,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [105]:
# com balanceamento automático
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [100]:
# sem balanceamento
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [128]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(class_weight={'Adequada': 1, 'Boa': 2,
                                              'Não adequada': 2},
                                n_jobs=-1, random_state=42, verbose=-1))])

In [104]:
# MÉTRICAS DE TREINO sem balanceamento
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7700976709241172
Train Precision:
0.7514335326258706
Train Recall:
0.7700976709241172
Train F1:
0.752598576156604

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.82      0.93      0.87     32935
         Boa       0.62      0.38      0.47      9436
Não adequada       0.55      0.50      0.52      5545

    accuracy                           0.77     47916
   macro avg       0.66      0.60      0.62     47916
weighted avg       0.75      0.77      0.75     47916

Train Confusion Matrix:
[[30579  1361   995]
 [ 4619  3549  1268]
 [ 1952   821  2772]]


In [107]:
# MÉTRICAS DE TREINO com balanceamento automático
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7156064780031722
Train Precision:
0.7765125275242073
Train Recall:
0.7156064780031722
Train F1:
0.7314564753023192

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.92      0.77      0.84     32935
         Boa       0.50      0.43      0.47      9436
Não adequada       0.39      0.85      0.53      5545

    accuracy                           0.72     47916
   macro avg       0.60      0.69      0.61     47916
weighted avg       0.78      0.72      0.73     47916

Train Confusion Matrix:
[[25488  3489  3958]
 [ 1935  4104  3397]
 [  281   567  4697]]


In [129]:
# MÉTRICAS DE TREINO com balanceamento manual
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7526296018031555
Train Precision:
0.7640876293860746
Train Recall:
0.7526296018031555
Train F1:
0.7577105234938298

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.88      0.85      0.87     32935
         Boa       0.48      0.54      0.51      9436
Não adequada       0.52      0.54      0.53      5545

    accuracy                           0.75     47916
   macro avg       0.63      0.65      0.64     47916
weighted avg       0.76      0.75      0.76     47916

Train Confusion Matrix:
[[27915  3658  1362]
 [ 2934  5131  1371]
 [  698  1830  3017]]


In [103]:
# MÉTRICAS DE TESTE sem balanceamento
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7424874791318865

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.81      0.91      0.86      8234
         Boa       0.53      0.34      0.41      2360
Não adequada       0.47      0.42      0.45      1386

    accuracy                           0.74     11980
   macro avg       0.61      0.56      0.57     11980
weighted avg       0.72      0.74      0.72     11980


Confusion Matrix:
[[7513  415  306]
 [1216  797  347]
 [ 515  286  585]]


In [108]:
# MÉTRICAS DE TESTE com balanceamento automático
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6885642737896495

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.91      0.76      0.83      8234
         Boa       0.44      0.38      0.41      2360
Não adequada       0.35      0.77      0.48      1386

    accuracy                           0.69     11980
   macro avg       0.57      0.64      0.57     11980
weighted avg       0.75      0.69      0.71     11980


Confusion Matrix:
[[6287  906 1041]
 [ 539  900  921]
 [  94  230 1062]]


In [130]:
# MÉTRICAS DE TESTE com balanceamento manual
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7235392320534224

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.87      0.83      0.85      8234
         Boa       0.44      0.50      0.47      2360
Não adequada       0.44      0.46      0.45      1386

    accuracy                           0.72     11980
   macro avg       0.58      0.60      0.59     11980
weighted avg       0.74      0.72      0.73     11980


Confusion Matrix:
[[6858  972  404]
 [ 789 1179  392]
 [ 198  557  631]]


Durante os experimentos, observou-se que as classes Atenção e Crítica apresentavam forte sobreposição, tanto nas métricas dos modelos quanto na análise exploratória e na clusterização. Mesmo após diferentes estratégias de balanceamento, recorte temporal e ajuste de pesos, o principal erro permaneceu concentrado entre essas duas classes. Por isso, optou-se por agrupá-las em uma única classe chamada “Não adequada”, representando situações que exigem acompanhamento técnico. Essa abordagem é mais coerente com o objetivo do sistema, que é apoiar a triagem inicial e a tomada de decisão do gestor, e não substituir a validação de uma equipe técnica especializada.

# Resultados do LightGBM com o Novo Rótulo

Após a reformulação do rótulo em três classes, foram realizados novos experimentos com o algoritmo **LightGBM**, utilizando o intervalo temporal de **2000 a 2008**. Essa etapa teve como objetivo avaliar se a nova estrutura de classificação seria mais adequada ao comportamento dos dados e se reduziria os problemas observados anteriormente entre as classes **Atenção** e **Crítica**.

A nova classificação utilizada foi composta por três categorias:

```text
Score 5 → Adequada
Score 4 → Boa
Score 0, 1, 2 ou 3 → Não adequada
```

Essa alteração foi motivada pelos resultados anteriores, nos quais os modelos apresentavam dificuldade persistente em distinguir amostras classificadas como **Atenção** e **Crítica**. Ao agrupar essas duas categorias em uma única classe, buscou-se representar melhor o papel do modelo dentro do sistema AquaSense: atuar como uma ferramenta de triagem inicial, identificando registros que merecem atenção técnica, sem necessariamente afirmar uma condição crítica definitiva.

## Estratégias avaliadas

Foram testadas três configurações do LightGBM:

1. Modelo sem balanceamento;
2. Modelo com `class_weight="balanced"`;
3. Modelo com pesos manuais, atribuindo:

   * Adequada = 1;
   * Boa = 1;
   * Não adequada = 2.

A comparação entre essas abordagens foi feita com base na acurácia, precisão, recall, F1-score e análise das matrizes de confusão.

## Modelo sem balanceamento

O modelo sem balanceamento apresentou a maior acurácia no conjunto de teste, com aproximadamente **74%**. A classe **Adequada** obteve desempenho elevado, com precisão de **81%**, recall de **91%** e F1-score de **86%**.

Entretanto, as classes **Boa** e **Não adequada** apresentaram desempenho mais limitado. A classe **Não adequada** obteve precisão de **47%**, recall de **42%** e F1-score de **45%**.

Esse resultado mostra que, sem balanceamento, o modelo ainda tende a favorecer a classe majoritária, classificando com maior facilidade as amostras adequadas.

## Modelo com `class_weight="balanced"`

Ao utilizar o balanceamento automático, o modelo passou a atribuir maior importância às classes menos representadas. Como consequência, a classe **Não adequada** apresentou recall elevado, chegando a **77%** no conjunto de teste.

No entanto, essa melhora no recall veio acompanhada de uma queda expressiva na precisão, que ficou em **35%**. Isso indica que o modelo passou a classificar muitas amostras como **Não adequada**, aumentando o número de falsos positivos.

Além disso, a acurácia geral caiu para aproximadamente **69%**, mostrando que o balanceamento automático foi agressivo demais para esse cenário.

## Modelo com balanceamento manual

O melhor equilíbrio entre as métricas foi observado no modelo com pesos manuais:

```text
Adequada = 1
Boa = 1
Não adequada = 2
```

Nesse experimento, a acurácia no teste foi de aproximadamente **72%**, com uma distribuição mais equilibrada entre precisão e recall da classe **Não adequada**.

A classe **Não adequada** apresentou:

```text
Precisão = 44%
Recall = 46%
F1-score = 45%
```

Diferente do balanceamento automático, o modelo não aumentou excessivamente o recall às custas da precisão. Em vez disso, apresentou um desempenho mais estável e coerente com o objetivo do sistema.

A classe **Boa** também apresentou melhora em relação ao modelo sem balanceamento, atingindo recall de **50%** e F1-score de **47%** no conjunto de teste.

## Interpretação geral

Os resultados indicam que a reformulação do rótulo para três classes tornou o problema de classificação mais consistente com a estrutura dos dados. A antiga separação entre **Atenção** e **Crítica** gerava uma sobreposição elevada e dificultava a aprendizagem dos modelos. Com a nova classe **Não adequada**, o modelo passou a trabalhar com uma categoria mais representativa e mais alinhada ao objetivo prático do AquaSense.

Mesmo assim, os resultados mostram que a classificação ainda possui limitações. A classe **Não adequada** não apresenta métricas tão altas quanto a classe **Adequada**, o que indica que os padrões associados à inadequação da água continuam sendo mais difíceis de identificar. Isso pode estar relacionado à complexidade ambiental dos dados, à sobreposição entre condições intermediárias e à limitação das variáveis disponíveis para treinamento.

Apesar disso, os resultados obtidos com o balanceamento manual mostram um avanço importante em relação aos experimentos anteriores. A diferença entre precisão e recall foi reduzida, o modelo não apresentou indícios relevantes de overfitting e as métricas ficaram mais equilibradas entre treino e teste.

## Conclusão

Entre as estratégias avaliadas, o LightGBM com balanceamento manual apresentou o melhor compromisso entre desempenho geral e equilíbrio entre as classes.

A nova estrutura em três classes se mostrou mais adequada ao contexto do projeto, pois permite que o modelo atue como uma ferramenta de apoio à decisão, indicando se uma amostra está **Adequada**, **Boa** ou **Não adequada**. Essa abordagem é mais compatível com a proposta do AquaSense, que busca automatizar uma triagem inicial da qualidade da água e apoiar o gestor na identificação de situações que exigem análise técnica posterior.


## Salvando novo rótulo

In [131]:
df["conama_status"] = df["conama_status_v4"]

In [144]:
df = df.drop(
    columns=[
        "conama_status_original",
        "conama_status_v1",
        "conama_status_v2",
        "conama_status_v3",
        "conama_status_adjusted",
        "conama_status_v4",
        "environmental_score_original",
        "environmental_score_adjusted",
        "critical_failures"
    ],
    errors="ignore"
)

In [145]:
df["conama_status"].value_counts()

,count
conama_status,
Adequada,41169
Boa,11796
Não adequada,6931


In [146]:
df.shape

(59896, 23)

In [147]:
df.columns.tolist()

['Country',
 'Area',
 'Waterbody Type',
 'Date',
 'Ammonia (mg/l)',
 'Biochemical Oxygen Demand (mg/l)',
 'Dissolved Oxygen (mg/l)',
 'Orthophosphate (mg/l)',
 'pH (ph units)',
 'Temperature (cel)',
 'Nitrogen (mg/l)',
 'Nitrate (mg/l)',
 'CCME_Values',
 'CCME_WQI',
 'ph_ok',
 'od_ok',
 'dbo_ok',
 'nitrate_ok',
 'ammonia_limit',
 'ammonia_ok',
 'environmental_score',
 'conama_status',
 'Year']

In [148]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Boa,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Boa,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Boa,2003


In [149]:
df.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008_novorotulo.parquet",
    index=False
)